In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
print(df.shape)
print(df.columns)
print(df.dtypes)

(891, 15)
Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='str')
survived          int64
pclass            int64
sex                 str
age             float64
sibsp             int64
parch             int64
fare            float64
embarked            str
class          category
who                 str
adult_male         bool
deck           category
embark_town         str
alive               str
alone              bool
dtype: object


In [4]:
print(df.isna().sum())
print(df.nunique())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
survived         2
pclass           3
sex              2
age             88
sibsp            7
parch            7
fare           248
embarked         3
class            3
who              3
adult_male       2
deck             7
embark_town      3
alive            2
alone            2
dtype: int64


## Dataset: Titanic

- **Source:** Built-in seaborn sample dataset (`sns.load_dataset("titanic")`)
- **Original source:** Kaggle / historical Titanic passenger records
- **License:** Public domain / freely available for educational use via seaborn
- **Why I chose it:** It has a good mix of numeric columns (age, fare) and categorical columns (sex, class, embarked), over 200 rows (891 total), and a clear real-world question to explore — what affected passenger survival.

**Schema summary:**
- 891 rows, 15 columns
- Missing values: `age` (177 missing), `deck` (688 missing), `embarked`/`embark_town` (2 missing each)
- Mix of int, float, bool, category, and text columns

In [5]:
# 'deck' is missing for 688 of 891 rows (~77%) — too sparse to impute reliably, so we drop the column
df = df.drop(columns=["deck"])

# 'age' has 177 missing values — fill with the median age rather than dropping rows, to preserve sample size
df["age"] = df["age"].fillna(df["age"].median())

# 'embarked' and 'embark_town' are each missing only 2 values — safe to drop those few rows
df = df.dropna(subset=["embarked", "embark_town"])

print(df.shape)
print(df.isna().sum())

(889, 14)
survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64


In [6]:
print("Shape before groupby:", df.shape)

# groupby: compute average fare and average survival rate per class
class_stats = df.groupby("class").agg(
    avg_fare=("fare", "mean"),
    survival_rate=("survived", "mean"),
)

# merge: attach these class-level stats back onto every passenger's row
df = df.merge(class_stats, on="class", how="left")

print("Shape after merge:", df.shape)
df.head()

Shape before groupby: (889, 14)
Shape after merge: (889, 16)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,avg_fare,survival_rate
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False,13.675550,0.242363
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False,84.193516,0.626168
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True,13.675550,0.242363
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False,84.193516,0.626168
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True,13.675550,0.242363


In [7]:
print("Shape before pivot:", df.shape)

# pivot: reshape to see survival rate broken down by both class AND sex at once
# this answers a more specific question than groupby alone could: does the survival
# advantage of being in 1st class hold for men and women equally?
survival_pivot = df.pivot_table(
    values="survived",
    index="class",
    columns="sex",
    aggfunc="mean"
)

print("Pivot table shape:", survival_pivot.shape)
survival_pivot

Shape before pivot: (889, 16)
Pivot table shape: (3, 2)


sex,female,male
class,,
First,0.967391,0.368852
Second,0.921053,0.157407
Third,0.500000,0.135447


In [8]:
# pull the fare column into a NumPy array
fare_array = df["fare"].to_numpy()

# vectorized standardization (z-score): (value - mean) / std, applied to the whole array at once, no loop
fare_mean = fare_array.mean()
fare_std = fare_array.std()
fare_zscore = (fare_array - fare_mean) / fare_std

print("Mean:", fare_mean)
print("Std:", fare_std)
print("Min z-score:", fare_zscore.min())
print("Max z-score:", fare_zscore.max())

# assign the result back as a new column
df["fare_zscore"] = fare_zscore
df[["fare", "fare_zscore"]].head()

Mean: 32.09668087739032
Std: 49.669545099689564
Min z-score: -0.6462044460638905
Max z-score: 9.668550782149426


,fare,fare_zscore
0,7.2500,-0.500240
1,71.2833,0.788947
2,7.9250,-0.486650
3,53.1000,0.422861
4,8.0500,-0.484133


In [9]:
print(survival_by_class)

NameError: name 'survival_by_class' is not defined

In [ ]:
survival_by_class = df.groupby("class")["survived"].mean()
print(survival_by_class)

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(survival_by_class.index.astype(str), survival_by_class.values, color=["#4C72B0", "#DD8452", "#55A868"])
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate")
plt.title("First-class passengers survived at over 2.5x the rate of third-class")
plt.savefig("../reports/a2_chart1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
survived = df[df["survived"] == 1]
died = df[df["survived"] == 0]

plt.scatter(died["age"], died["fare"], alpha=0.5, label="Did not survive", color="#C44E52")
plt.scatter(survived["age"], survived["fare"], alpha=0.5, label="Survived", color="#55A868")

plt.xlabel("Age")
plt.ylabel("Fare")
plt.title("Survivors skew toward higher fares, regardless of age")
plt.legend()
plt.savefig("../reports/a2_chart2.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("Average fare - survived:", df[df["survived"] == 1]["fare"].mean())
print("Average fare - did not survive:", df[df["survived"] == 0]["fare"].mean())

In [ ]:
df.to_csv("../data/processed/titanic_cleaned.csv", index=False)
print("Saved:", df.shape)